# End-to-End Structured Regression Project

This notebook is the place where the course transitions from little toy examples to an actual project. The dataset is synthetic but realistic: equipment sales, seasonal patterns, promotion impact, and a clear business target.


## Define the business problem

We are predicting total equipment sales revenue from time and operational features. This is a realistic tabular problem with structured data, not a toy classification task.


In [ ]:
import pandas as pd
import numpy as np

rng = np.random.RandomState(7)
months = pd.date_range('2020-01-01', periods=240, freq='MS')
store_id = np.repeat(np.arange(1, 11), 24)
traffic = rng.randint(200, 700, size=240)
price = rng.uniform(500, 2000, size=240)
promo = rng.binomial(1, 0.25, size=240)
season = np.sin(np.arange(240) / 12)
revenue = 1200 + 0.9 * traffic + 56 * promo + 140 * season + rng.normal(0, 80, size=240)
df = pd.DataFrame({
    'month': months,
    'store_id': store_id,
    'traffic': traffic,
    'price': price,
    'promo': promo,
    'seasonal_signal': season,
    'revenue': revenue,
})
print(df.head())
assert len(df) == 240


## Clean and inspect the data

The most important lesson in a real regression project is that a realistic feature set needs careful inspection. We check missing values, ranges, and whether the target has useful variation.


In [ ]:
print(df.isna().sum().to_dict())
print(df['revenue'].describe())
assert df['revenue'].notna().all()


## Split by time to avoid leakage

A time-based split mimics the real-world deployment problem: we train on earlier months and test on later months.


In [ ]:
df = df.sort_values('month').reset_index(drop=True)
train = df.iloc[:180]
test = df.iloc[180:]
print(len(train), len(test))
assert len(test) == 60


## Train a benchmark model

This notebook introduces a comparison between a simple model and a more flexible one. The point is not to choose the most complex model but to understand when complexity is justified.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train = train[['traffic', 'price', 'promo', 'seasonal_signal']]
y_train = train['revenue']
X_test = test[['traffic', 'price', 'promo', 'seasonal_signal']]
y_test = test['revenue']
linear = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
rf = RandomForestRegressor(random_state=42, n_estimators=200, max_depth=8).fit(X_train, y_train)
for name, model in [('linear', linear), ('rf', rf)]:
    pred = model.predict(X_test)
    print(name, round(mean_absolute_error(y_test, pred), 2), round(r2_score(y_test, pred), 3))


## Short submission rubric

The final block is the capstone-style narrative: explain what the model predicts, what features matter, and what limitations remain. This is where students move from code to a business recommendation.


In [ ]:
best = rf
preds = best.predict(X_test)
print('final MAE:', round(mean_absolute_error(y_test, preds), 2))
print('final R2:', round(r2_score(y_test, preds), 3))
assert len(preds) == len(y_test)
